In [189]:
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import s3fs

## Dataset Import

In [194]:
# Import FIP Dataset

s3_path_fip = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "publish/data-product/financial_inventory_projection_report_network_update/"
)

dataset = ds.dataset(
    s3_path_fip,
    format="parquet",
    partitioning="hive" 
)

table = dataset.to_table(
    filter=(
        ds.field("date").isin(["202612"])  # "202712", "202812" 
    ) & (
        ds.field("snapshot_date") >= "2025-10-01"
    ) & ~(
        (ds.field("snapshot_date") == "2026-01-23") &
        (ds.field("snapshot_type") == "friday")
    )
)


df_fip = table.to_pandas()
df_fip.head()

,material,plant,date,quantity,total_cost,concost_source,unit_of_measure,cost_per_unit,source,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,snapshot_type,snapshot_date
0,CRU CRU D-40-Ref,1037,202612,0.000000,NaN,missing,None,NaN,rr,None,None,None,None,None,None,None,None,friday,2025-10-03
1,1382955Z0,1734,202612,0.175000,NaN,missing,None,NaN,rr,nan,HALB,CLINICAL,API,API / DRUG SUBSTANCE,nan,00000nan,PHARMA,friday,2025-10-03
2,1234398,1760,202612,473.619466,146053.880553,concost dp,KG,308.37812,rr,Ipilimumab (Yervoy),RAW,COMMERCIAL,RAW MATERIAL,RAW MATERIAL,nan,03301503,BIOLOGICS,friday,2025-10-03
3,1457248,2061,202612,561.000000,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-10-03
4,1457516,2061,202612,34000.000000,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-10-03


In [195]:
df_fip.shape

(649034, 19)

In [196]:
# Import Plant Type Data

s3_path_plants = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "bms_internal_vs_external_plants/"
    "bms_internal_vs_external_plants.parquet"
)

df_plants = pd.read_parquet(s3_path_plants)
#df_plants.head()

In [197]:
# Import node type Dataset

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files_nt = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/"
    "src__sap_t001w/data/*.parquet"
)

df_ntype = pd.read_parquet(
    parquet_files_nt,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_ntype.head()

In [198]:
# Import Material Master Data

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "sap_material_master/data/*.parquet"
)

df_mm = pd.read_parquet(
    parquet_files,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_mm.head()

In [199]:
# import boto3

# s3 = boto3.client("s3")

# bucket = "m3-intel-hub-dp-us-east-1-517292-prod"
# prefix = "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/src__sap_t001w"

# response = s3.list_objects_v2(
#     Bucket=bucket,
#     Prefix=prefix
# )

# if "Contents" in response:
#     for obj in response["Contents"]:
#         print(obj["Key"], obj["Size"])
# else:
#     print("No objects found or no access.")

## Data Prep

In [200]:
# Create has_non_zero flag at material–plant level

df_fip["has_non_zero"] = (
    df_fip
    .groupby(["material", "plant"])["total_cost"]
    .transform(lambda x: (x != 0).any())
    .astype(int)
)

# Apply the filter
df_fip = df_fip.loc[df_fip["has_non_zero"] == 1].drop(columns="has_non_zero")

In [201]:
df_fip.shape

(417617, 19)

In [202]:
base1 = df_fip.copy()

In [203]:
# Join material master
mm_cols = [
    "material_number",
    "plant",
    "profit_center",
    #"corporate_brand",
    "brand_name",
    #"corp_brand_id",
    "material_description",
    #"material_type",
    "material_group",
    "old_material_number",
    "base_unit_of_measure",
    "unit_of_weight",
    #"development_lifecycle_status",
    "plant_specific_material_status",
    "mrp_type",
    "procurement_type",
    "safety_stock",
    "minimum_lot_size",
    "maximum_lot_size",
    "fixed_lot_size",
    "total_replenishment_lead_time",
    "total_shelf_life",
    "batch_management",
    "abc_indicator",
    #"valuation_class",
    #"price_unit",
    #"price_control_indicator",
]

material_master_sel = (
    df_mm[mm_cols]
    .drop_duplicates(subset=["material_number", "plant"])
)


base1 = base1.merge(
    material_master_sel,
    left_on=["material", "plant"],
    right_on=["material_number", "plant"],
    how="left",
    validate="m:1" 
)
base1 = base1.drop(columns=["material_number"])

base1 = base1.merge(
    df_plants[["Plant", "Plant Type"]],
    left_on=["plant"],
    right_on=["Plant"],
    how="left"
).drop(columns=["Plant"])

node_type_lkp = (
    df_ntype[["werks", "nodetype"]]
    .drop_duplicates(subset=["werks"])
)

base1 = base1.merge(
    node_type_lkp,
    left_on="plant",
    right_on="werks",
    how="left",
    validate="m:1"
).drop(columns=["werks"])

base1.shape

(417617, 39)

In [204]:
# rearrange columns for better readability
new_cols = [
    'corporate_brand',
    'material_type', 'development_lifecycle_status', 'enterprise_category',
    'enterprise_sub_category', 'dosage_form_parent', 'corp_brand_id',
    'network_or_business_unit',
    'profit_center', 'brand_name', 'material_description', 'material_group',
    'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
    'plant_specific_material_status', 'mrp_type', 'procurement_type',
    'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
    'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
    'batch_management', 'abc_indicator',
    'source', 'concost_source', 'Plant Type', 'nodetype',
    'unit_of_measure',
    'material', 'plant', 'date', 'quantity', 'total_cost',
    'cost_per_unit', 'snapshot_type', 'snapshot_date'
]

base1 = base1[new_cols]


In [205]:
base1.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name', 'material_description', 'material_group',
       'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
       'plant_specific_material_status', 'mrp_type', 'procurement_type',
       'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
       'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
       'batch_management', 'abc_indicator', 'source', 'concost_source',
       'Plant Type', 'nodetype', 'unit_of_measure', 'material', 'plant',
       'date', 'quantity', 'total_cost', 'cost_per_unit', 'snapshot_type',
       'snapshot_date'],
      dtype='object')

In [206]:
# Add material entry flag 

base = base1.copy()
first_seen = (
    base.groupby(["material", "plant","date"])["snapshot_date"]
      .transform("min")
)

base["sku_status"] = np.where(
    base["snapshot_date"] == first_seen,
    "NEW",
    "EXISTING"
)


In [207]:
# fip copy df for data prep
df = base.copy()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])


# snapshot lookup table
snapshot_calendar = (
    df[["snapshot_type", "snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)


# attach prev snapshot to snapshot calendar
snapshot_calendar["prev_snapshot_date"] = (
    snapshot_calendar
    .groupby("snapshot_type")["snapshot_date"]
    .shift(1)
)
snapshot_calendar      # comparing bd13 - bd13 snapshots and friday-friday snapshots. no bd13-fri snapshots

,snapshot_type,snapshot_date,prev_snapshot_date
36332,bd13,2025-10-17,NaT
136176,bd13,2025-11-19,2025-10-17
237531,bd13,2025-12-17,2025-11-19
358786,bd13,2026-01-23,2025-12-17
0,friday,2025-10-03,NaT
16428,friday,2025-10-10,2025-10-03
56226,friday,2025-10-24,2025-10-10
76191,friday,2025-10-31,2025-10-24
96241,friday,2025-11-07,2025-10-31
116226,friday,2025-11-14,2025-11-07


In [208]:
# Attach previous snapshot date to each row

df = df.merge(
    snapshot_calendar[["snapshot_type", "snapshot_date", "prev_snapshot_date"]],
    on=["snapshot_type", "snapshot_date"],
    how="left"
)

In [209]:
# Prepare current and previous frames

# Current snapshot frame
current_df = df.copy()

current_df = current_df.rename(columns={
    "quantity": "quantity_curr",
    "cost_per_unit": "cost_per_unit_curr",
    "total_cost": "total_cost_curr",
})

# Previous snapshot frame
previous_df = df.rename(columns={
    "snapshot_date": "snapshot_date_prev",
    "quantity": "quantity_prev",
    "cost_per_unit": "cost_per_unit_prev",
    "total_cost": "total_cost_prev",
})[
    [
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
        "quantity_prev",
        "cost_per_unit_prev",
        "total_cost_prev",
    ]
]


# Join current to previous snapshot
rca_base = current_df.merge(
    previous_df,
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    how="left"
)

In [210]:
rca_base.shape

(417859, 45)

In [211]:
# Flags for material–plant presence 

# 1. Identify NEW in current snapshot
# Present now, but not present in previous snapshot
rca_base["is_new_in_current_snapshot"] = (
    rca_base["prev_snapshot_date"].notna() &
    rca_base["quantity_prev"].isna()
)


# 2. Identify DROPPED in current snapshot
# Present in previous snapshot but missing in current snapshot

# Identify valid previous snapshots (calendar-safe)
valid_prev_snapshots = (
    snapshot_calendar["prev_snapshot_date"]
        .dropna()
        .unique()
)

# Restrict previous snapshot data to valid transitions
previous_df_valid = previous_df[
    previous_df["snapshot_date_prev"].isin(valid_prev_snapshots)
]

# Anti-join: rows present in previous but missing in current
prev_only = previous_df_valid.merge(
    current_df[
        [
            "material",
            "plant",
            "date",
            "snapshot_type",
            "prev_snapshot_date",
        ]
    ],
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    how="left",
    indicator=True
).query("_merge == 'left_only'")

# Mark dropped rows
prev_only["is_dropped_in_current_snapshot"] = True

# Ensure column alignment for concat
for col in rca_base.columns:
    if col not in prev_only.columns:
        prev_only[col] = np.nan

# Current snapshot rows are NOT dropped
rca_base["is_dropped_in_current_snapshot"] = False


# 3. Combine current + dropped rows

final_rca_frame = pd.concat(
    [rca_base, prev_only[rca_base.columns]],
    ignore_index=True
)

# 4. Normalize boolean flags

for col in [
    "is_new_in_current_snapshot",
    "is_dropped_in_current_snapshot",
]:
    final_rca_frame[col] = (
        final_rca_frame[col]
            .replace({1: True, 0: False})
            .fillna(False)
            .astype("boolean")
    )

/tmp/ipykernel_326592/3310939321.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rca_frame = pd.concat(
/tmp/ipykernel_326592/3310939321.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [212]:
final_rca_frame.shape

(424603, 47)

In [213]:
snapshot_mapping_check = (
    final_rca_frame
    .loc[:, ["snapshot_type", "snapshot_date", "prev_snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)

snapshot_mapping_check

,snapshot_type,snapshot_date,prev_snapshot_date
36364,bd13,2025-10-17,NaT
136383,bd13,2025-11-19,2025-10-17
237773,bd13,2025-12-17,2025-11-19
359028,bd13,2026-01-23,2025-12-17
418277,bd13,NaT,NaT
0,friday,2025-10-03,NaT
16428,friday,2025-10-10,2025-10-03
56258,friday,2025-10-24,2025-10-10
76293,friday,2025-10-31,2025-10-24
96413,friday,2025-11-07,2025-10-31


Error handling

1. Division by zero & invalid math -
    Previous quantity = 0
    Previous cost = 0
    Previous FIP = 0
    Volatility = 0
2. Min rolling window
   less than min periods of 3
   Newly introduced SKU
   Flat history causing volatility = 0
3. double counting due to duplicates
4. missing prev snapshot data - nulls
   SKU appears for first time
   SKU disappears and reappears
   Previous quantity / cost not available
5. Extreme values in the history
    Very large quantities or costs in the past pushing the present numbers. The outliers in the past affects the current numbers. Exlcude the historical outliers
   

# Data Transformations

### Driver Calculations

In [214]:
# Step 1: Compute raw change metrics (always runs)

# Base deltas

final_rca_frame["delta_quantity"] = (
    final_rca_frame["quantity_curr"] - final_rca_frame["quantity_prev"]
)

final_rca_frame["delta_cost_per_unit"] = (
    final_rca_frame["cost_per_unit_curr"] - final_rca_frame["cost_per_unit_prev"]
)


# Impact decomposition

# Quantity Impact 
final_rca_frame["quantity_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["cost_per_unit_prev"]
)

# Cost Impact
final_rca_frame["cost_impact"] = (
    final_rca_frame["delta_cost_per_unit"] * final_rca_frame["quantity_prev"]
)

# Intercation
final_rca_frame["interaction_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["delta_cost_per_unit"]
)

# Total Change
final_rca_frame["total_fip_change"] = (
    final_rca_frame["quantity_impact"] +
    final_rca_frame["cost_impact"] +
    final_rca_frame["interaction_impact"]
)

# Core Metrics

# delta_quantity_pct 
final_rca_frame["delta_quantity_pct"] = np.where(
    final_rca_frame["quantity_prev"] > 0,
    final_rca_frame["delta_quantity"] / final_rca_frame["quantity_prev"],
    np.nan
)

# delta_cost_per_unit_pct

final_rca_frame["delta_cost_per_unit_pct"] = np.where(
    final_rca_frame["cost_per_unit_prev"] > 0,
    final_rca_frame["delta_cost_per_unit"] / final_rca_frame["cost_per_unit_prev"],
    np.nan
)


# Contribution shares (absolute, normalized)
impact_abs_sum_qc = (
    final_rca_frame["quantity_impact"].abs() +
    final_rca_frame["cost_impact"].abs()
)

impact_abs_sum_all = (
    impact_abs_sum_qc +
    final_rca_frame["interaction_impact"].abs()
)


# quantity_impact_pct_of_total 
final_rca_frame["quantity_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["quantity_impact"].abs() / impact_abs_sum_qc,
    0
)

# cost_impact_pct_of_total 
final_rca_frame["cost_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["cost_impact"].abs() / impact_abs_sum_qc,
    0
)

# interaction_pct
final_rca_frame["interaction_pct"] = np.where(
    impact_abs_sum_all > 0,
    final_rca_frame["interaction_impact"].abs() / impact_abs_sum_all,
    0
)


# Dominance Score

final_rca_frame["abs_qty_impact"] = final_rca_frame["quantity_impact"].abs()
final_rca_frame["abs_cost_impact"] = final_rca_frame["cost_impact"].abs()

final_rca_frame["dominance_score"] = np.where(
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"]) == 0,
    0.0,  # avoid divide-by-zero → treat as neutral
    (final_rca_frame["abs_qty_impact"] - final_rca_frame["abs_cost_impact"]) /
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"])
)

### Data Sufficiency Metrics

In [215]:
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["is_qty_present_greater0"] = (
    final_rca_frame["quantity_curr"] > 0
).astype(int)

# 1) history_weeks_count
final_rca_frame["history_weeks_count_greater0"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["is_qty_present_greater0"]
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=[0,1,2], drop=True)
)

final_rca_frame["hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant", "date"])["quantity_curr"]
        .transform(lambda x: x.notna().cumsum() - 1)
)

final_rca_frame["has_sufficient_6periods"] = (
    final_rca_frame["hist_count"] >= 6
)


# 2) presence_stability_score
final_rca_frame["presence_stability_score"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)

# 3) History confidence (intentionally mirrors stability)
final_rca_frame["history_confidence"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)


# 4) sku_presence_class
conditions = [
    final_rca_frame["is_new_in_current_snapshot"] == True,  
    final_rca_frame["is_dropped_in_current_snapshot"] == True,
    (
        (final_rca_frame["quantity_prev"] == 0) &
        (final_rca_frame["quantity_curr"] > 0) &
        (final_rca_frame["history_weeks_count_greater0"] < 6)
    ),
    (
        (final_rca_frame["presence_stability_score"] >= 0.75) &
        (final_rca_frame["history_weeks_count_greater0"] >= 6)
    )
]
 
choices = [
    "ENTERED",
    "OUT",
    "REACTIVATED",
    "STABLE"
]
 
final_rca_frame["sku_presence_class"] = np.select(
    conditions,
    choices,
    default="ERRATIC"
)


# 5) uom_change_flag
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["unit_of_measure_prev"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["base_unit_of_measure"]
    .shift(1)
)

final_rca_frame["uom_change_flag"] = (
    final_rca_frame["unit_of_measure_prev"].notna() &
    (final_rca_frame["unit_of_measure_prev"] != final_rca_frame["unit_of_measure"])
)


# 6) duplicate_sku_plant_flag
dup_counts = (
    final_rca_frame
    .groupby(["material", "plant","date", "snapshot_date"])
    .size()
    .rename("dup_count")
    .reset_index()
)
final_rca_frame = final_rca_frame.merge(
    dup_counts,
    on=["material", "plant","date", "snapshot_date"],
    how="left"
)
final_rca_frame["duplicate_sku_plant_flag"] = (
    final_rca_frame["dup_count"] > 1
)


# 7) dominance_instability_flag

final_rca_frame["dominance_score_var_4"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["dominance_score"]
    .rolling(window=4, min_periods=2)
    .var()
    .reset_index(level=[0,1,2], drop=True)
)
final_rca_frame["dominance_instability_flag"] = (
    final_rca_frame["dominance_score_var_4"] > 0.25
)


# 8) RCA Mode

conditions = [
    # 1) No RCA: UOM change or duplicate SKU–plant
    (
        final_rca_frame["uom_change_flag"].fillna(False).astype(bool) |
        final_rca_frame["duplicate_sku_plant_flag"].fillna(False).astype(bool)
    ),
    # 2) Entry / Exit RCA
    (
        final_rca_frame["sku_presence_class"]
        .isin(["ENTERED", "OUT"])
        .fillna(False)
    ),
    # 3) Limited RCA: insufficient history
    (
        ~final_rca_frame["has_sufficient_6periods"]
    ),
    # 4) Full temporal RCA: stable SKU
    (
        final_rca_frame["sku_presence_class"]
        .eq("STABLE")
        .fillna(False)
    )
]

choices = [
    "NO_RCA",
    "ENTRY_EXIT_RCA",
    "LIMITED_RCA",
    "FULL_TEMPORAL_RCA"
]

final_rca_frame["rca_mode"] = np.select(
    conditions,
    choices,
    default="LIMITED_RCA"
)

### Noise vs Signal Determination 

In [216]:
# Sorting

final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

#  1) Business materiality (value-based)   
# How big the change is in value terms, relative to prior fip.
final_rca_frame["relative_fip_impact"] = np.where(
    final_rca_frame["total_cost_prev"] > 0,
    final_rca_frame["total_fip_change"].abs() /
    final_rca_frame["total_cost_prev"],
    np.nan
)

#### Persistance

In [217]:
# Temporal persistence (directional consistency)

final_rca_frame["quantity_change_sign"] = np.sign(
    final_rca_frame["delta_quantity"]
) 

def persistence_score(series):
    score = []
    current = 0
    prev = 0
    for v in series:
        if v == 0 or pd.isna(v):
            current = 0
        elif v == prev:
            current += 1
        else:
            current = 1
        score.append(current)
        prev = v
    return score

final_rca_frame["quantity_persistence_score"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["quantity_change_sign"]
    .transform(persistence_score)
)


In [218]:
# Outlier Identification

# PARAMETERS

ROLLING_WINDOW = 12
MIN_PERIODS = 6

EXTREME_PCT_CHANGE = 0.5      # 50%
LOW_LEVEL_FLOOR = 0.05        # collapse threshold (5%)
HIGH_LEVEL_MULT = 10         # spike threshold (10x)

ROBUST_Z_THRESHOLD = 3
MIN_ZSCORE_POINTS = 3
PCTL_FALLBACK = 0.95

# 1) Rolling median of quantity (baseline scale)

final_rca_frame["quantity_rolling_median_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .median()
        )
)

# 2) Structural extreme change (scale-based)

final_rca_frame["is_structural_extreme_qty"] = (
    (final_rca_frame["delta_quantity_pct"].abs() >= EXTREME_PCT_CHANGE) &
    (
        (final_rca_frame["quantity_curr"] <=
         LOW_LEVEL_FLOOR * final_rca_frame["quantity_rolling_median_12w"]) |
        (final_rca_frame["quantity_curr"] >=
         HIGH_LEVEL_MULT * final_rca_frame["quantity_rolling_median_12w"])
    )
)



# 3) CLEAN delta series (absolute, exclude structural extremes)

final_rca_frame["clean_delta_quantity"] = final_rca_frame["delta_quantity"]

final_rca_frame.loc[
    final_rca_frame["is_structural_extreme_qty"],
    "clean_delta_quantity"
] = np.nan



# 4) Robust rolling MAD of absolute delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 5) Robust MAD-based z-score (absolute delta)

def robust_zscore(series, min_points=MIN_ZSCORE_POINTS):
    valid = series.dropna()

    if len(valid) < min_points:
        return pd.Series(np.nan, index=series.index)

    median = np.nanmedian(valid)
    mad = np.nanmedian(np.abs(valid - median))

    if mad == 0 or np.isnan(mad):
        return pd.Series(np.nan, index=series.index)

    return (series - median) / (1.4826 * mad)


final_rca_frame["quantity_z_scr"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(lambda x: robust_zscore(x.shift(1)))
)



# 6) Z-score availability (CORRECT, LOCAL gating)

final_rca_frame["qty_z_hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=1)
                      .count()
        )
)

final_rca_frame["has_quantity_zscore"] = (
    (final_rca_frame["qty_z_hist_count"] >= MIN_ZSCORE_POINTS) &
    (final_rca_frame["delta_qty_rolling_mad_12w"] > 0)
)



# 7) Statistical outlier (MAD-based)

final_rca_frame["is_statistical_outlier_qty"] = (
    final_rca_frame["has_quantity_zscore"] &
    (final_rca_frame["quantity_z_scr"].abs() > ROBUST_Z_THRESHOLD)
)


# 8) Percentile fallback (ONLY when z-score unavailable)
final_rca_frame["qty_abs_pct_threshold"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .abs()
                      .quantile(PCTL_FALLBACK)
        )
)

# Fallback scale must be valid (non-zero)
final_rca_frame["has_valid_pct_scale"] = (
    final_rca_frame["qty_abs_pct_threshold"] > 0
)

final_rca_frame["is_pct_outlier_qty"] = (
    final_rca_frame["clean_delta_quantity"].abs() >
    final_rca_frame["qty_abs_pct_threshold"]
)

# 9) FINAL quantity outlier flag (hierarchical & SAFE)

final_rca_frame["is_quantity_outlier"] = (
    final_rca_frame["is_structural_extreme_qty"] |
    np.where(
        final_rca_frame["has_quantity_zscore"],
        final_rca_frame["is_statistical_outlier_qty"],
        final_rca_frame["has_valid_pct_scale"] &
        final_rca_frame["is_pct_outlier_qty"]
    )
)


#### Change Point Detection

In [219]:
# Change Point Detection

# PARAMETERS

CP_LONG_WINDOW = 12
CP_LONG_MIN = 6
CP_SHORT_WINDOW = 6
CP_SHORT_MIN = 3

CP_TREND_STRENGTH_THRESHOLD = 2
CP_FALLBACK_STRENGTH_THRESHOLD = 5   # conservative
MIN_PERSISTENCE_CP = 2


# 1) Rolling average of ABSOLUTE delta quantity (long window)

final_rca_frame["rolling_avg_delta_qty_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .mean()
        )
)


# 2) rolling MAD of ABSOLUTE delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 3) Primary trend strength (MAD-based)

final_rca_frame["trend_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    (1.4826 * final_rca_frame["delta_qty_rolling_mad_12w"])
)

# Invalidate degenerate cases
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_mad_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_mad_12w"].isna()),
    "trend_strength"
] = np.nan

final_rca_frame["has_valid_cp_strength"] = (
    final_rca_frame["trend_strength"].notna()
)


# 4) GENERIC fallback scale (never collapses)

final_rca_frame["delta_qty_rolling_median_abs_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(lambda s: np.nanmedian(np.abs(s)), raw=True)
        )
)

final_rca_frame["fallback_change_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    final_rca_frame["delta_qty_rolling_median_abs_12w"]
)

# Invalidate fallback when scale unusable
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_median_abs_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_median_abs_12w"].isna()),
    "fallback_change_strength"
] = np.nan

final_rca_frame["use_fallback_cp"] = (
    final_rca_frame["trend_strength"].isna() &
    final_rca_frame["fallback_change_strength"].notna()
)

# 5) Short-window trend confirmation (ABSOLUTE delta)
final_rca_frame["rolling_avg_delta_qty_6w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_SHORT_WINDOW, min_periods=CP_SHORT_MIN)
                      .mean()
        )
)

final_rca_frame["trend_confirmed"] = (
    np.sign(final_rca_frame["rolling_avg_delta_qty_12w"]) ==
    np.sign(final_rca_frame["rolling_avg_delta_qty_6w"])
)


# 6) FINAL generic change-point detection
final_rca_frame["change_point_detected"] = (
    (final_rca_frame["quantity_persistence_score"] >= MIN_PERSISTENCE_CP) &
    final_rca_frame["trend_confirmed"] &
    (
        # Primary MAD-based path
        (
            final_rca_frame["has_valid_cp_strength"] &
            (final_rca_frame["trend_strength"] > CP_TREND_STRENGTH_THRESHOLD)
        )
        |
        # Fallback absolute-scale path
        (
            final_rca_frame["use_fallback_cp"] &
            (final_rca_frame["fallback_change_strength"] > CP_FALLBACK_STRENGTH_THRESHOLD)
        )
    )
)



#### Regime Stability

In [220]:
# Regime stability - validate the change point with stability

REGIME_STABILITY_WINDOW = 3        # how many snapshots must settle
REGIME_STABILITY_TOLERANCE = 0.2   # ±20% band around new level

# 1) Reference "new level" after change

final_rca_frame["post_cp_level"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .shift(1)
)


# 2) Within-band check
final_rca_frame["within_new_regime_band"] = (
    final_rca_frame["quantity_curr"].between(
        final_rca_frame["post_cp_level"] * (1 - REGIME_STABILITY_TOLERANCE),
        final_rca_frame["post_cp_level"] * (1 + REGIME_STABILITY_TOLERANCE)
    )
)


# 3) Rolling stability confirmation
final_rca_frame["regime_stability_score"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["within_new_regime_band"]
        .transform(
            lambda x: x.shift(-1)   # look forward (post-change validation)
                      .rolling(REGIME_STABILITY_WINDOW, min_periods=REGIME_STABILITY_WINDOW)
                      .sum()
        )
)

final_rca_frame["is_regime_stable"] = (
    final_rca_frame["regime_stability_score"] >= REGIME_STABILITY_WINDOW
)


final_rca_frame["effective_change_point"] = (
    final_rca_frame["change_point_detected"] &
    final_rca_frame["is_regime_stable"]
)


#### Noise & Signal Flagging

In [221]:
# BASE METHOD 

final_rca_frame["is_noise_base"] = (
    final_rca_frame["is_quantity_outlier"] &
    (~final_rca_frame["effective_change_point"])
)

# insufficient data → cannot classify as noise yet
final_rca_frame.loc[
    ~final_rca_frame["has_sufficient_6periods"],
    "is_noise_base"
] = False

final_rca_frame["is_signal_base"] = ~final_rca_frame["is_noise_base"]


In [222]:
# 2nd Layer Flaging

final_rca_frame["is_explainable"] = (
    # data quality must be OK
    (~final_rca_frame["uom_change_flag"]) &
    (~final_rca_frame["duplicate_sku_plant_flag"]) 
    # &
    # # must not be lifecycle noise -- needs data backed thresholds to handle different edge cases
    # (final_rca_frame["sku_presence_class"] == "STABLE")
)

final_rca_frame["is_noise_governed"] = (
    final_rca_frame["is_noise_base"] |
    (~final_rca_frame["is_explainable"])
)

final_rca_frame["is_signal_governed"] = (
    ~final_rca_frame["is_noise_governed"]
)

final_rca_frame["noise_reason"] = None

final_rca_frame.loc[
    final_rca_frame["uom_change_flag"] |
    final_rca_frame["duplicate_sku_plant_flag"],
    "noise_reason"
] = "DATA_QUALITY"

final_rca_frame.loc[
    final_rca_frame["noise_reason"].isna() &
    final_rca_frame["is_noise_base"],
    "noise_reason"
] = "STATISTICAL_NOISE"


## Price Outliers Addition

In [223]:
PRICE_OUTLIER_PCT_THRESHOLD = 1   # 100% change

final_rca_frame["is_price_outlier"] = (
    final_rca_frame["delta_cost_per_unit_pct"].abs() >= PRICE_OUTLIER_PCT_THRESHOLD
)

final_rca_frame["price_event_type"] = "NO_PRICE_EVENT"

# Price-driven RCA case (quantity is NOT the driver)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    (~final_rca_frame["is_quantity_outlier"]),
    "price_event_type"
] = "PRICE_PRIMARY"

# Mixed RCA case (price amplifies a quantity-driven change)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    final_rca_frame["is_quantity_outlier"],
    "price_event_type"
] = "PRICE_SECONDARY"


#### Brand + Snapshot Date Level Top 10 SKUs

In [224]:
final_rca_frame["abs_fip_change"] = final_rca_frame["total_fip_change"].abs()

final_rca_frame["snapshot_signal_rank"] = (
    final_rca_frame
    .where(final_rca_frame["is_signal_governed"])
    .groupby(["snapshot_date", "corporate_brand","date"])["abs_fip_change"]
    .rank(method="first", ascending=False)
)
final_rca_frame["is_top10_contributor"] = (
    final_rca_frame["snapshot_signal_rank"] <= 10
)

final_rca_frame = final_rca_frame.drop(["snapshot_signal_rank","abs_fip_change"],axis=1)

In [225]:
final_rca_frame.head()

,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,profit_center,brand_name,...,effective_change_point,is_noise_base,is_signal_base,is_explainable,is_noise_governed,is_signal_governed,noise_reason,is_price_outlier,price_event_type,is_top10_contributor
0,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
1,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
2,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
3,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
4,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False


In [226]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#     # ,
#     #"ABRAXANE"
#     #,
#     #"REVLIMID"
# ]
# dates_to_keep = ["202612"] #, "202712", "202812"]

# filtered_df = final_rca_frame[
#     final_rca_frame["corporate_brand"].isin(brands_to_keep) &
#     final_rca_frame["date"].isin(dates_to_keep)
# ]

In [227]:
# final_rca_frame.loc[
#     final_rca_frame["corporate_brand"].str.contains(
#         "All Other Pharmaceut", case=False, na=False
#     ),
#     "corporate_brand"
# ].unique()


In [228]:
final_rca_frame.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name',
       ...
       'effective_change_point', 'is_noise_base', 'is_signal_base',
       'is_explainable', 'is_noise_governed', 'is_signal_governed',
       'noise_reason', 'is_price_outlier', 'price_event_type',
       'is_top10_contributor'],
      dtype='object', length=113)

# Brand & Material Type Level Aggregations

### Data Prep

In [235]:
sku_df = final_rca_frame.copy()

# Define SKU correctly: material–plant
sku_df["sku_id"] = (
    sku_df["material"].astype(str) + "||" +
    sku_df["plant"].astype(str)
)

# Normalize boolean (critical for stability)
sku_df["is_signal_governed"] = (
    sku_df["is_signal_governed"]
        .fillna(False)
        .astype(bool)
)

# Helper columns
sku_df["abs_sku_impact"] = sku_df["total_fip_change"].abs()

sku_df["signal_impact"] = np.where(
    sku_df["is_signal_governed"],
    sku_df["abs_sku_impact"],
    0.0
)

sku_df["conflict_evaluable"] = (
    sku_df["is_signal_governed"] &
    sku_df["quantity_impact"].notna() &
    sku_df["cost_impact"].notna()
)

sku_df["driver_conflict"] = np.where(
    sku_df["conflict_evaluable"],
    np.sign(sku_df["quantity_impact"]) != np.sign(sku_df["cost_impact"]),
    np.nan
)

sku_df["weighted_dom_component"] = (
    sku_df["dominance_score"] * sku_df["abs_sku_impact"]
)

sku_df["signal_structural_cp"] = (
    sku_df["is_signal_governed"] &
    sku_df["effective_change_point"]
)


In [268]:
sku_df.shape

(424603, 120)

# Hierarchy based Sorting

In [271]:
# 1. Hierarchy definition

HIERARCHY = [
    ["material_type"],
    ["dosage_form_parent"],
    ["material_group"],
    ["material"],
    ["nodetype"],
    ["Plant Type"],
    ["plant"],
    ["material", "plant"]  # material–plant (most granular)
]

# 2. Compute explainability at a level (SKU table ONLY)

def compute_explainability(sku_brand_df, group_cols):
    """
    Explainability = share of brand's total absolute impact
    explained by the top contributor at this level.
    """

    group_keys = ["date", "snapshot_date"] + group_cols

    grp = (
        sku_brand_df
        .groupby(group_keys, as_index=False)
        .agg(level_abs_impact=("abs_sku_impact", "sum"))
    )

    total_impact = sku_brand_df["abs_sku_impact"].sum()

    if total_impact == 0 or grp.empty:
        return 0.0, {}

    top_row = grp.loc[grp["level_abs_impact"].idxmax()]
    top_impact = top_row["level_abs_impact"]

    explainability = top_impact / total_impact

    top_entity = {col: top_row[col] for col in group_cols}

    return explainability, top_entity


# 3. Walk hierarchy & find dominant level

def find_dominant_level(sku_brand_df):
    """
    Walks hierarchy and selects the level
    that provides the maximum marginal clarity gain.
    """

    results = []
    prev_explainability = 0.0

    for level in HIERARCHY:
        explainability, entity = compute_explainability(sku_brand_df, level)

        marginal_gain = explainability - prev_explainability

        results.append({
            "level": " × ".join(level),
            "explainability": explainability,
            "marginal_gain": marginal_gain,
            "entity": entity
        })

        prev_explainability = explainability

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[results_df["marginal_gain"].idxmax()]

    return {
        "dominant_level": best_row["level"],
        "dominant_entity": best_row["entity"],
        "dominant_explainability": best_row["explainability"],
        "full_trace": results_df
    }


# 4. Apply per brand + date + snapshot

dominant_results = []

group_keys = ["corporate_brand","corp_brand_id", "date", "snapshot_date"]

for (brand, brand_id, date, snapshot_date), sku_brand_df in sku_df.groupby(group_keys):

    result = find_dominant_level(sku_brand_df)

    dominant_results.append({
        "corporate_brand": brand,
        "corp_brand_id": brand_id,
        "date": date,
        "snapshot_date": snapshot_date,
        "dominant_level": result["dominant_level"],
        "dominant_entity": result["dominant_entity"],
        "dominant_explainability": result["dominant_explainability"]
    })

brand_dominance_df = pd.DataFrame(dominant_results)

brand_dominance_df.head()


,corporate_brand,corp_brand_id,date,snapshot_date,dominant_level,dominant_entity,dominant_explainability
0,CM BCM Golcadomide,08101516,202612,2025-10-03,material_type,{},0.0
1,CM BCM Golcadomide,08101516,202612,2025-10-10,material_type,{},0.0
2,CM BCM Golcadomide,08101516,202612,2025-10-17,material_type,{},0.0
3,CM BCM Golcadomide,08101516,202612,2025-10-24,material_type,{},0.0
4,CM BCM Golcadomide,08101516,202612,2025-10-31,material_type,{},0.0


In [272]:
brand_dominance_df.tail()

,corporate_brand,corp_brand_id,date,snapshot_date,dominant_level,dominant_entity,dominant_explainability
3868,recothrom,00251126,202612,2025-11-07,material_type,{},0.0
3869,recothrom,00251126,202612,2025-11-14,material_type,{},0.0
3870,recothrom,00251126,202612,2025-11-19,material_type,{},0.0
3871,recothrom,00251126,202612,2025-11-21,material_type,{},0.0
3872,recothrom,00251126,202612,2025-11-28,material_type,{},0.0


### Brand Level Calculations along with Dominant Level

In [273]:
LEVEL_MAP = {
    "material_type": {
        "cols": ["material_type"],
        "level_name": "BRAND_MATERIAL_TYPE"
    },
    "dosage_form_parent": {
        "cols": ["dosage_form_parent"],
        "level_name": "BRAND_DOSAGE_FORM"
    },
    "material_group": {
        "cols": ["material_group"],
        "level_name": "BRAND_MATERIAL_GROUP"
    },
    "material": {
        "cols": ["material"],
        "level_name": "BRAND_MATERIAL"
    },
    "nodetype": {
        "cols": ["nodetype"],
        "level_name": "BRAND_NODETYPE"
    },
    "Plant Type": {
        "cols": ["Plant Type"],
        "level_name": "BRAND_PLANT_TYPE"
    },
    "plant": {
        "cols": ["plant"],
        "level_name": "BRAND_PLANT"
    },
    "material × plant": {
        "cols": ["material", "plant"],
        "level_name": "BRAND_MATERIAL_PLANT"
    }
}


In [274]:
def compute_entity_metrics(df):
    out = {}

    out["ΔFIP"] = df["total_fip_change"].sum()
    out["Prior_FIP"] = df["total_cost_prev"].sum()
    out["Abs_Impact"] = df["abs_sku_impact"].sum()

    out["Total_SKUs"] = df["sku_id"].nunique()
    out["Signal_SKUs"] = df["is_signal_governed"].sum()
    out["Signal_Impact"] = df["signal_impact"].sum()

    out["Qty_Impact"] = df["quantity_impact"].sum()
    out["Cost_Impact"] = df["cost_impact"].sum()

    out["Weighted_Dominance"] = (
        df["weighted_dom_component"].sum() /
        df["abs_sku_impact"].sum()
        if df["abs_sku_impact"].sum() > 0 else np.nan
    )

    out["Driver_Conflict_%"] = (
        df["driver_conflict"].sum() / out["Signal_SKUs"]
        if out["Signal_SKUs"] > 0 else np.nan
    )

    out["Avg_Persistence"] = df["quantity_persistence_score"].mean()

    out["Structural_Shift_Index"] = (
        (df["signal_structural_cp"].sum() / out["Signal_SKUs"]) * 100
        if out["Signal_SKUs"] > 0 else np.nan
    )

    # Ratios
    out["FIP_pct_change"] = (
        out["ΔFIP"] / out["Prior_FIP"]
        if out["Prior_FIP"] != 0 else np.nan
    )

    out["Net_vs_Abs_Ratio"] = (
        abs(out["ΔFIP"]) / out["Abs_Impact"]
        if out["Abs_Impact"] > 0 else np.nan
    )

    out["Signal_SKU_%"] = (
        out["Signal_SKUs"] / out["Total_SKUs"]
        if out["Total_SKUs"] > 0 else np.nan
    )

    out["Impact_from_Signal_%"] = (
        out["Signal_Impact"] / out["Abs_Impact"]
        if out["Abs_Impact"] > 0 else np.nan
    )

    out["Qty_Impact_%"] = (
        out["Qty_Impact"] / out["ΔFIP"]
        if out["ΔFIP"] != 0 else np.nan
    )

    out["Cost_Impact_%"] = (
        out["Cost_Impact"] / out["ΔFIP"]
        if out["ΔFIP"] != 0 else np.nan
    )

    out["Ownership_Clarity_Index"] = abs(
        out["Qty_Impact_%"] - out["Cost_Impact_%"]
    )

    return out


In [275]:
# Define Null columns
def null_metrics_by_level(row):
    lvl = row["level"]

    if lvl == "BRAND_MATERIAL_PLANT":
        row[["Top_5_SKU_%", "Top_10_SKU_%"]] = np.nan

    return row

# build dominant-entity rows and append
dominant_rows = []

for _, dom in brand_dominance_df.iterrows():

    # Dominance strength check
    if dom["dominant_explainability"] < 0.5:
        continue

    cfg = LEVEL_MAP.get(dom["dominant_level"])
    if cfg is None:
        continue

    mask = (
        (sku_df["corp_brand_id"] == dom["corp_brand_id"]) &
        (sku_df["date"] == dom["date"]) &
        (sku_df["snapshot_date"] == dom["snapshot_date"])
    )

    for c, v in dom["dominant_entity"].items():
        mask &= (sku_df[c] == v)

    sub_df = sku_df.loc[mask]
    if sub_df.empty:
        continue

    metrics = compute_entity_metrics(sub_df)

    brand_fip_series = brand_df.loc[
        (brand_df["corp_brand_id"] == dom["corp_brand_id"]) &
        (brand_df["date"] == dom["date"]) &
        (brand_df["snapshot_date"] == dom["snapshot_date"]),
        "ΔFIP"
    ]

    if brand_fip_series.empty:
        continue

    brand_fip = brand_fip_series.iloc[0]

    metrics.update({
        "corp_brand_id": dom["corp_brand_id"],
        "corporate_brand": (
            f"{dom['corporate_brand']}_"
            + "_".join(dom["dominant_entity"].values())
        ),
        "date": dom["date"],
        "snapshot_date": dom["snapshot_date"],
        "level": cfg["level_name"],
        "Entity_Impact_onBrandlevel_%": (
            metrics["ΔFIP"] / brand_fip
            if brand_fip != 0 else np.nan
        )
    })

    dominant_rows.append(metrics)


dominant_entity_df = pd.DataFrame(dominant_rows)
dominant_entity_df = dominant_entity_df.apply(null_metrics_by_level, axis=1)

# Append to brand_df
final_agg_df = pd.concat([brand_df, dominant_entity_df], ignore_index=True)

In [277]:
brand_df["corporate_brand"].nunique()


1

In [246]:
# Select & rename final columns

final_df = (
    final_agg_df
    .rename(columns={
        "corporate_brand": "entity",
        "FIP_pct_change": "FIP_change_pct",
        "Entity_Impact_onBrandlevel_%": "Entity_Impact_onBrandlevel_pct",
        "Net_vs_Abs_Ratio": "Net_vs_Abs_pct",
        "Signal_SKU_%": "Signal_SKU_pct",
        "Impact_from_Signal_%": "Impact_from_Signal_pct",
        "Top_1_SKU_%": "Top_1_SKU_pct",
        "Top_5_SKU_%": "Top_5_SKU_pct",
        "Top_10_SKU_%": "Top_10_SKU_pct",
        "Plant_Concentration_%": "Plant_Concentration_pct",
        "Qty_Impact_%": "Qty_Impact_pct",
        "Cost_Impact_%": "Cost_Impact_pct",
        "Driver_Conflict_%": "Driver_Conflict_pct",
        "Actionability_Index": "Actionability_Index_pct",
        "Ownership_Clarity_Index": "Ownership_Clarity_Index_pct",
        "Explainability_Score": "Explainability_Score_pct",
        "Weighted_Dominance": "Driver_Weighted_Dominance",
        "Structural_Shift_Index": "Structural_Shift_Index_pct"
    })
    [
        [
            "corp_brand_id",
            "entity",
            "level",
            "date",
            "snapshot_date",
            "FIP_change_pct",
            "Abs_Impact",
            "Entity_Impact_onBrandlevel_pct",
            "Net_vs_Abs_pct",
            "Signal_SKU_pct",
            "Impact_from_Signal_pct",
            "Top_1_SKU_pct",
            "Top_5_SKU_pct",
            "Top_10_SKU_pct",
            "Plant_Concentration_pct",
            "Qty_Impact_pct",
            "Cost_Impact_pct",
            "Driver_Weighted_Dominance",
            "Driver_Conflict_pct",
            "Avg_Persistence",
            "Structural_Shift_Index_pct",
            "Actionability_Index_pct",
            "Ownership_Clarity_Index_pct",
            "Explainability_Score_pct"
            
        ]
    ]
)


# Percentage columns (×100)
percent_cols = [
    "FIP_change_pct",
    "Entity_Impact_onBrandlevel_pct",
    "Net_vs_Abs_pct",
    "Signal_SKU_pct",
    "Impact_from_Signal_pct",
    "Top_1_SKU_pct",
    "Top_5_SKU_pct",
    "Top_10_SKU_pct",
    "Plant_Concentration_pct",
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Driver_Conflict_pct",
    "Actionability_Index_pct",
    "Ownership_Clarity_Index_pct",
    "Explainability_Score_pct",
]

final_df[percent_cols] = (
    final_df[percent_cols]
    .astype(float)
    .mul(100)
    .round(2)
)


# Numeric rounding

# Round most numeric columns to 2 decimals
numeric_round_2_cols = [
    "Abs_Impact",
    "Avg_Persistence",
    "Structural_Shift_Index_pct",
]

final_df[numeric_round_2_cols] = (
    final_df[numeric_round_2_cols]
    .astype(float)
    .round(2)
)

# Round Driver_Weighted_Dominance to 0 decimals
final_df["Driver_Weighted_Dominance"] = (
    final_df["Driver_Weighted_Dominance"]
    .astype(float)
    .round(0)
    .astype("Int64")
)


def remove_negative_zero(x):
    if isinstance(x, (int, float, np.floating)) and np.isclose(x, 0):
        return 0.0
    return x

cols_to_clean = [
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Net_vs_Abs_pct",
    "Driver_Conflict_pct",
]

final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


/tmp/ipykernel_326592/1630615981.py:122: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


In [269]:
final_df.shape

(55, 24)

In [248]:
final_df[final_df['snapshot_date'] == '2026-01-09']

,corp_brand_id,entity,level,date,snapshot_date,FIP_change_pct,Abs_Impact,Entity_Impact_onBrandlevel_pct,Net_vs_Abs_pct,Signal_SKU_pct,...,Plant_Concentration_pct,Qty_Impact_pct,Cost_Impact_pct,Driver_Weighted_Dominance,Driver_Conflict_pct,Avg_Persistence,Structural_Shift_Index_pct,Actionability_Index_pct,Ownership_Clarity_Index_pct,Explainability_Score_pct
29,03302101,REVLIMID,BRAND,202612,2026-01-09,-97.93,1.493309e+09,NaN,99.91,86.82,...,99.46,100.0,0.0,1,16.28,0.29,0.27,0.22,100.0,75.52
50,03302101,REVLIMID_HALB,BRAND_MATERIAL_TYPE,202612,2026-01-09,-99.49,1.486367e+09,99.58,99.96,81.65,...,NaN,100.0,0.0,1,26.97,0.51,0.00,NaN,100.0,NaN


In [350]:
# final_agg_df.to_csv('3brands_agg_file.csv')

In [421]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#      ,
#     "ABRAXANE"
#     ,
#     "REVLIMID"
# ]
# dates_to_keep = ["202612", "202712", "202812"]

# fil = final_agg_df[
#     final_agg_df["corporate_brand"].isin(brands_to_keep) &
#     final_agg_df["date"].isin(dates_to_keep)
# ]

In [425]:
# final_agg_df.to_csv('Allbrands_agg_file.csv')